# E3 · Target encoding before the cross-validation: a transformer inside the Pipeline

**Type**: Find the bug · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

Fix an evaluation inflated by target encoding run on the whole dataset before the cross-validation, by writing a
scikit-learn compatible transformer (`fit`, `transform`, `fit_transform`) and using it inside a `Pipeline`.

## Dataset

`ds.loans(with_missing=False)`: the `occupation` column (40 levels, with a true effect) and `noise`, a category with 300 random levels
with no link to `defaulted`. The functions `te_estimate`, `te_apply`, `te_oof` (those of exercise A3) are already
defined in the setup cell.

## Educational purpose

Close the circle of A3: know **where** to put an encoder that uses the target. A transformer inside a `Pipeline` receives
`fit_transform(X, y)` on the training folds and only `transform(X)` on the validation folds: it is the only place where target
encoding does not leak the target.

## Task

1. Run the colleague's cell: it encodes all the rows with `te_estimate` + `te_apply` **before** the cross-validation, then evaluates a `LogisticRegression`. Save `auc_bug` (it is very high).
2. Write `MyTargetEncoder(BaseEstimator, TransformerMixin)` with `__init__(self, m=5)`, `fit(X, y)` (save the mappings for every column of `X`, a DataFrame), `transform(X)` (apply the mappings; unknown categories → global mean) and `fit_transform(X, y)` which returns the **out-of-fold** encoding of the training set (use `te_oof`) after calling `fit`.
3. Build `make_pipeline(MyTargetEncoder(m=5), LogisticRegression())` and evaluate it with the same `cv`: save `auc_ok`.
4. Repeat the same pipeline on the `noise` column alone: `noise_only_auc` (it should stay close to 0.5).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

df = ds.loans(with_missing=False)
X = df[["occupation"]].copy()
X["noise"] = np.array([f"r{i}" for i in np.random.default_rng(1).integers(0, 300, len(df))])
y = df["defaulted"].to_numpy()
cv = StratifiedKFold(5, shuffle=True, random_state=0)


def te_estimate(cat, y, m):
    d = pd.DataFrame({"c": np.asarray(cat), "y": np.asarray(y, dtype=float)})
    g = d.groupby("c")["y"].agg(["sum", "count"])
    overall = d["y"].mean()
    return ((g["sum"] + m * overall) / (g["count"] + m)).to_dict(), overall


def te_apply(cat, mapping, overall):
    return pd.Series(np.asarray(cat)).map(mapping).fillna(overall).to_numpy(dtype=float)


def te_oof(cat, y, m, k=5, seed=0):
    cat, y = np.asarray(cat), np.asarray(y)
    out = np.zeros(len(cat))
    for tr, va in KFold(k, shuffle=True, random_state=seed).split(cat):
        mapping, overall = te_estimate(cat[tr], y[tr], m)
        out[va] = te_apply(cat[va], mapping, overall)
    return out

## Your work

### The colleague's code

In [ ]:
# encoding of the WHOLE dataset with the mapping computed on the WHOLE dataset, before the cross-validation
Z = np.column_stack([te_apply(X[c], *te_estimate(X[c], y, 5)) for c in X.columns])
auc_bug = cross_val_score(LogisticRegression(), Z, y, cv=cv, scoring="roc_auc").mean()
print(f"colleague's cross-validated AUC: {auc_bug:.3f}")

### Your transformer

In [ ]:
class MyTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, m=5):
        self.m = m

    def fit(self, X, y):
        ...
        return self

    def transform(self, X):
        ...

    def fit_transform(self, X, y):
        ...

In [ ]:
auc_ok = auc_bug
noise_only_auc = auc_bug

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='In `fit`: `self.columns_ = list(X.columns)` and `self.maps_ = {c: te_estimate(X[c], y, self.m) for c in self.columns_}`. In `transform`: `np.column_stack([te_apply(X[c], *self.maps_[c]) for c in self.columns_])`. The `fit_transform`: call `self.fit(X, y)` and then `np.column_stack([te_oof(X[c], y, self.m) for c in self.columns_])`.'):
    check(auc_bug > 0.75, "the colleague's number is very high: it is the target leaking", "auc_bug should exceed 0.75: don't modify the colleague's code")
    check(0.55 < auc_ok < 0.72, "with the pipeline the AUC drops to a realistic value (between 0.55 and 0.72)",
          "auc_ok must come from cross_val_score on make_pipeline(MyTargetEncoder(m=5), LogisticRegression())")
    check(noise_only_auc < 0.6, "on the noise alone the cross-validated AUC is close to 0.5", "on the noise alone with the pipeline the AUC must stay below 0.6")
    check(isinstance(MyTargetEncoder(), TransformerMixin), "the transformer is scikit-learn compatible", "inherit from BaseEstimator and TransformerMixin")
    enc = MyTargetEncoder(m=5).fit(X, y)
    new = enc.transform(pd.DataFrame({"occupation": ["occ_zz"], "noise": ["r_new"]}))
    check(np.isfinite(new).all() and close(new, [[y.mean(), y.mean()]]), "unknown categories get the global mean",
          "a never-seen category must be encoded with the global mean of the target")
    check(not np.allclose(MyTargetEncoder(m=5).fit_transform(X, y), enc.transform(X)),
          "fit_transform (out-of-fold) is different from fit + transform on the same training set",
          "fit_transform must use te_oof; transform uses the mapping of the whole training set")
done()

---
The commented solution is in `solutions/E_find_the_bug/E3_target_encoding_before_cv.ipynb`. Open it after you pass the check, or after you have really tried.